# Staged-to-real fall detection: experiment notebook

Do fall detectors trained on **staged lab falls** work on **real falls**?
Run the steps in order. Everything is saved to your Google Drive in `fall-cvpr/`, so if Colab disconnects, re-run **Setup** and continue where you stopped.

**Before you start:** *Runtime → Change runtime type → T4 GPU*.

| Step | What it does | GPU? | Time |
|---|---|---|---|
| Setup | connect Drive, download code | – | 1 min |
| 1. Mine real clips | streams the 48 GB OOPS! dataset once, scores ~11k clips, keeps ~3,600 short clips | yes | a few hours (speed is printed) |
| 2. Label | you mark clips as *fall / not a fall* | no | about 10 s per clip |
| 3. Staged data | downloads and encodes UR Fall | yes | ~5 min |
| 4. Experiment | trains and evaluates, writes the results table | no | 1 min |

## Setup

In [ ]:
import subprocess, torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU for steps 1 and 3")
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/fall-cvpr'
!mkdir -p {OUT}
!rm -rf /content/fall-detector && git clone -q -b claude/beautiful-brown-gdjzcq https://github.com/apersonli55555/fall-detector.git /content/fall-detector
!pip -q install open_clip_torch av
import sys; sys.path.insert(0, '/content/fall-detector/vision')
print("ready")

## Step 1: mine real-world clips from OOPS! (GPU)
Safe to stop and restart: it saves every 500 videos and skips what's done. It still has to re-stream the archive from the start, but skipped videos go fast.
It keeps two kinds of clips: a **random sample of about 600** (the fair test set) and the **top 3,000** the AI rates most fall-like (extra training data).

In [ ]:
!python /content/fall-detector/vision/oops_mine.py --out {OUT}/oops

## Step 2: label clips (no GPU needed)
To save GPU quota you can switch to a CPU runtime here; re-run **Setup** afterwards.

**What counts as a fall:** a **person** (not an animal) unintentionally ends up on the ground, floor or a lower level, and it happens in the clip. If you can't tell, press *Unsure*.

Label **all the test-set clips first** (the counter shows progress; they come first automatically). Then label as many training-pool clips as you like; a few hundred is enough to start.
Every click is saved immediately, so you can stop any time.

In [ ]:
from labeler import Labeler
Labeler(f'{OUT}/oops').show()

## Step 3: staged lab falls (UR Fall dataset, GPU)

In [ ]:
!python /content/fall-detector/vision/urfall_prep.py --out {OUT}/urfall.npz --cache /content/urfall

## Step 4: run the experiment
Writes `results/staged_to_real.md` to your Drive. **Send me that table** (or the `.json` next to it) and I'll interpret it and plan the next step.

In [ ]:
!python /content/fall-detector/vision/experiment.py --staged {OUT}/urfall.npz --oops {OUT}/oops